In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("advanced_pyspark").getOrCreate()

In [3]:
from google.colab import files
uploaded = files.upload()

Saving plant_master.csv to plant_master.csv
Saving manufacturing_production_extra_12k.csv to manufacturing_production_extra_12k.csv
Saving manufacturing_production_120k.csv to manufacturing_production_120k.csv
Saving machine_master.csv to machine_master.csv
Saving machine_configurations.json to machine_configurations.json


## 1. Load the Data in PySpark

In [4]:
df = spark.read.csv("manufacturing_production_120k.csv", header=True, inferSchema=True)

#1
df.show(20)
#2
df.printSchema()
#3
print(df.columns)
#4
print(df.count())
#5
df.select("production_id", "plant_id", "machine_id", "product_family", "units_produced").show()
#6
print(df.dtypes)
#7
df.sample(fraction=0.0001, seed=42).show(10)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0000001|     2026-08-07|    PL03|     MC011|     OP0275|Evening|     MTR-100|        Motors|         1060|           993|             30|              34|    388.11|         61.4|          3.88|        Rework|              No|    Charan|
|    PR0000002|     

## 2. Creating DataFrames

In [5]:
from pyspark.sql.types import StructType, StructField, StringType

#1 list of tuples
plant_data = [
    ("PL01", "Chennai Motors Plant", "Chennai"),
    ("PL02", "Pune Controls Plant", "Pune"),
    ("PL03", "Hosur Pumps Plant", "Hosur"),
    ("PL04", "Coimbatore Assembly Plant", "Coimbatore"),
    ("PL05", "Bangalore Electronics Plant", "Bangalore"),
    ("PL06", "Hyderabad Components Plant", "Hyderabad")
]

#2 explicit column names
small_plant_df = spark.createDataFrame(plant_data, ["plant_id", "plant_name", "city"])
small_plant_df.show()

#3 StructType
plant_schema = StructType([
    StructField("plant_id", StringType(), True),
    StructField("plant_name", StringType(), True),
    StructField("city", StringType(), True)
])
small_plant_df2 = spark.createDataFrame(plant_data, plant_schema)
small_plant_df2.show()

#4 print schema
small_plant_df.printSchema()
small_plant_df2.printSchema()

#5 constant column
small_plant_df = small_plant_df.withColumn("country", F.lit("India"))
small_plant_df.show()

+--------+--------------------+----------+
|plant_id|          plant_name|      city|
+--------+--------------------+----------+
|    PL01|Chennai Motors Plant|   Chennai|
|    PL02| Pune Controls Plant|      Pune|
|    PL03|   Hosur Pumps Plant|     Hosur|
|    PL04|Coimbatore Assemb...|Coimbatore|
|    PL05|Bangalore Electro...| Bangalore|
|    PL06|Hyderabad Compone...| Hyderabad|
+--------+--------------------+----------+

+--------+--------------------+----------+
|plant_id|          plant_name|      city|
+--------+--------------------+----------+
|    PL01|Chennai Motors Plant|   Chennai|
|    PL02| Pune Controls Plant|      Pune|
|    PL03|   Hosur Pumps Plant|     Hosur|
|    PL04|Coimbatore Assemb...|Coimbatore|
|    PL05|Bangalore Electro...| Bangalore|
|    PL06|Hyderabad Compone...| Hyderabad|
+--------+--------------------+----------+

root
 |-- plant_id: string (nullable = true)
 |-- plant_name: string (nullable = true)
 |-- city: string (nullable = true)

root
 |-- plan

## 3. Pandas DataFrame

In [6]:
import pandas as pd

pdf = pd.read_csv("manufacturing_production_120k.csv")

#1
print(pdf.head())
#2
print(pdf.tail())
#3
print(pdf.shape)
#4
print(pdf.columns)
#5
print(pdf.dtypes)
#6
print(pdf[["plant_id", "machine_id", "units_produced"]])
#7
pdf_renamed = pdf.rename(columns={"units_produced": "production_output"})
print(pdf_renamed.head())
#8
print(pdf[pdf["quality_status"] == "Reject"])
#9
print(pdf[pdf["temperature_c"] > 75])
#10  (in plant_master.csv, Hyderabad plant = PL01)
print(pdf[(pdf["shift"] == "Night") & (pdf["plant_id"] == "PL01")])

  production_id production_date plant_id machine_id operator_id    shift  \
0     PR0000001      2026-08-07     PL03      MC011      OP0275  Evening   
1     PR0000002      2026-08-03     PL04      MC019      OP0489  Morning   
2     PR0000003      2026-05-20     PL01      MC002      OP0189  Evening   
3     PR0000004      2026-01-15     PL03      MC013      OP0076  Morning   
4     PR0000005      2026-09-21     PL04      MC020      OP0501  Morning   

  product_code product_family  units_planned  units_produced  defective_units  \
0      MTR-100         Motors           1060             993               30   
1      CTL-330    Controllers            809             836                0   
2      MTR-300         Motors            442             380               17   
3      PMP-210          Pumps            787             720                7   
4      PMP-110          Pumps            344             296                7   

   downtime_minutes  energy_kwh  temperature_c  vibratio

## 4. Dask DataFrame

In [7]:
import dask.dataframe as dd
import time

# assume_missing=True avoids dtype-inference errors when integer columns contain blanks
ddf = dd.read_csv("manufacturing_production_120k.csv", assume_missing=True)

#1
print(ddf.head())
#2
print("Default partitions:", ddf.npartitions)
ddf_small_blocks = dd.read_csv("manufacturing_production_120k.csv", blocksize="2MB", assume_missing=True)
print("Partitions with 2MB blocks:", ddf_small_blocks.npartitions)
#3
selected = ddf[["plant_id", "product_family", "units_produced"]]
print(selected.head())
#4
rejected = ddf[ddf["quality_status"] == "Reject"]
print(rejected.head())
#5
avg_units = ddf.groupby("plant_id")["units_produced"].mean()
#6
defects = ddf.groupby("product_family")["defective_units"].sum()
#7 .compute() triggers the actual (lazy) computation
print(avg_units.compute())
print(defects.compute())
print("Rejected rows:", len(rejected))
#8 Pandas vs Dask on the same aggregation
start = time.time()
pandas_result = pdf.groupby("product_family")["defective_units"].sum()
print("Pandas:", round(time.time() - start, 4), "seconds")
print(pandas_result)

start = time.time()
dask_result = ddf.groupby("product_family")["defective_units"].sum().compute()
print("Dask:", round(time.time() - start, 4), "seconds")
print(dask_result)

  production_id production_date plant_id machine_id operator_id    shift  \
0     PR0000001      2026-08-07     PL03      MC011      OP0275  Evening   
1     PR0000002      2026-08-03     PL04      MC019      OP0489  Morning   
2     PR0000003      2026-05-20     PL01      MC002      OP0189  Evening   
3     PR0000004      2026-01-15     PL03      MC013      OP0076  Morning   
4     PR0000005      2026-09-21     PL04      MC020      OP0501  Morning   

  product_code product_family  units_planned  units_produced  defective_units  \
0      MTR-100         Motors         1060.0           993.0             30.0   
1      CTL-330    Controllers          809.0           836.0              0.0   
2      MTR-300         Motors          442.0           380.0             17.0   
3      PMP-210          Pumps          787.0           720.0              7.0   
4      PMP-110          Pumps          344.0           296.0              7.0   

   downtime_minutes  energy_kwh  temperature_c  vibratio

## 5. Manipulating DataFrames

In [8]:
#1
df_mfg = df.withColumn("production_efficiency", F.round(F.col("units_produced") / F.col("units_planned") * 100, 2))
df_mfg.select("production_id", "units_planned", "units_produced", "production_efficiency").show()
#2
df_mfg = df_mfg.withColumn("good_units", F.col("units_produced") - F.col("defective_units"))
df_mfg.select("production_id", "units_produced", "defective_units", "good_units").show()
#3
df_mfg = df_mfg.withColumn("country", F.lit("India"))
df_mfg.select("production_id", "country").show()
#4
df_mfg = df_mfg.withColumnRenamed("energy_kwh", "energy_consumed")
df_mfg.select("production_id", "energy_consumed").show()
#5
df_mfg = df_mfg.withColumn("units_produced", F.col("units_produced").cast("double"))
df_mfg.printSchema()
#6
df_mfg = df_mfg.withColumn("defect_percentage", F.round(F.col("defective_units") / F.col("units_produced") * 100, 2))
df_mfg.select("production_id", "defective_units", "units_produced", "defect_percentage").show()
#7
df_mfg = df_mfg.withColumn("energy_per_unit", F.round(F.col("energy_consumed") / F.col("units_produced"), 4))
df_mfg.select("production_id", "energy_consumed", "units_produced", "energy_per_unit").show()
#8
df_mfg.select(
    "production_id", "production_efficiency", "good_units", "country",
    "energy_consumed", "units_produced", "defect_percentage", "energy_per_unit"
).show()

+-------------+-------------+--------------+---------------------+
|production_id|units_planned|units_produced|production_efficiency|
+-------------+-------------+--------------+---------------------+
|    PR0000001|         1060|           993|                93.68|
|    PR0000002|          809|           836|               103.34|
|    PR0000003|          442|           380|                85.97|
|    PR0000004|          787|           720|                91.49|
|    PR0000005|          344|           296|                86.05|
|    PR0000006|          958|           795|                82.99|
|    PR0000007|         1165|           955|                81.97|
|    PR0000008|          862|           895|               103.83|
|    PR0000009|          313|           289|                92.33|
|    PR0000010|          532|           527|                99.06|
|    PR0000011|          577|           512|                88.73|
|    PR0000012|          980|           968|                98

## 6. Dropping Data

In [9]:
#1
df_q1 = df.drop("supervisor")
df_q1.printSchema()
#2
df_q2 = df.drop("supervisor", "maintenance_flag")
print(df_q2.columns)
#3
df_q3 = df.dropDuplicates()
print(df_q3.count())
#4
df_q4 = df.dropDuplicates(["production_id"])
print(df_q4.count())
#5
df_q5 = df.dropna(subset=["operator_id"])
print(df_q5.count())
#6 drop only when BOTH are missing
df_q6 = df.dropna(how="all", subset=["temperature_c", "vibration_mm_s"])
print(df_q6.count())
#7
df.select("product_family").distinct().show()
#8
df.select("plant_id", "machine_id").distinct().orderBy("plant_id", "machine_id").show(40)

root
 |-- production_id: string (nullable = true)
 |-- production_date: date (nullable = true)
 |-- plant_id: string (nullable = true)
 |-- machine_id: string (nullable = true)
 |-- operator_id: string (nullable = true)
 |-- shift: string (nullable = true)
 |-- product_code: string (nullable = true)
 |-- product_family: string (nullable = true)
 |-- units_planned: integer (nullable = true)
 |-- units_produced: integer (nullable = true)
 |-- defective_units: integer (nullable = true)
 |-- downtime_minutes: integer (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- vibration_mm_s: double (nullable = true)
 |-- quality_status: string (nullable = true)
 |-- maintenance_flag: string (nullable = true)

['production_id', 'production_date', 'plant_id', 'machine_id', 'operator_id', 'shift', 'product_code', 'product_family', 'units_planned', 'units_produced', 'defective_units', 'downtime_minutes', 'energy_kwh', 'temperature_c', 'vibratio

## 7. Sorting

In [10]:
#1
df.orderBy(F.col("units_produced").asc()).show()
#2
df.orderBy(F.col("units_produced").desc()).show()
#3
df.orderBy(F.col("defective_units").desc()).show()
#4
df.orderBy(F.col("plant_id").asc(), F.col("units_produced").desc()).show()
#5
df.orderBy(F.col("units_produced").desc()).show(20)
#6
df.orderBy(F.col("downtime_minutes").desc_nulls_last()).show(10)
#7
df.groupBy("plant_id", "machine_id") \
    .agg(F.max("vibration_mm_s").alias("max_vibration")) \
    .orderBy(F.col("max_vibration").desc()) \
    .show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0038643|     2026-08-06|    PL02|     MC009|     OP0361|Evening|     MTR-100|        Motors|          250|           205|              6|              73|     65.18|        55.99|          2.72|          Pass|              No|    Harini|
|    PR0059668|     

## 8. Aggregations

In [11]:
#1
df.select(F.sum("units_planned").alias("total_units_planned")).show()
#2
df.select(F.sum("units_produced").alias("total_units_produced")).show()
#3
df.select(F.sum("defective_units").alias("total_defective_units")).show()
#4
df.select(F.avg("units_produced").alias("avg_units_produced")).show()
#5
df.select(F.max("units_produced").alias("max_production")).show()
#6
df.select(F.min("units_produced").alias("min_production")).show()
#7
df.select(F.avg("downtime_minutes").alias("avg_downtime")).show()
#8
df.select(F.sum("energy_kwh").alias("total_energy_kwh")).show()
#9
df.select(F.avg("temperature_c").alias("avg_temperature")).show()
#10
df.select(F.avg("vibration_mm_s").alias("avg_vibration")).show()

+-------------------+
|total_units_planned|
+-------------------+
|           87094621|
+-------------------+

+--------------------+
|total_units_produced|
+--------------------+
|            80927888|
+--------------------+

+---------------------+
|total_defective_units|
+---------------------+
|              2030244|
+---------------------+

+------------------+
|avg_units_produced|
+------------------+
| 674.3990666666666|
+------------------+

+--------------+
|max_production|
+--------------+
|          1245|
+--------------+

+--------------+
|min_production|
+--------------+
|           205|
+--------------+

+-----------------+
|     avg_downtime|
+-----------------+
|33.93149474428351|
+-----------------+

+--------------------+
|    total_energy_kwh|
+--------------------+
|2.8368014080000117E7|
+--------------------+

+------------------+
|   avg_temperature|
+------------------+
|63.021532392167266|
+------------------+

+-----------------+
|    avg_vibration|
+----------

## 9. groupBy()

In [12]:
#1
df.groupBy("plant_id").agg(F.sum("units_produced").alias("total_production")).orderBy(F.desc("total_production")).show()
#2
df.groupBy("product_family").agg(F.sum("units_produced").alias("total_production")).orderBy(F.desc("total_production")).show()
#3
df.groupBy("plant_id").agg(F.sum("defective_units").alias("total_defects")).orderBy(F.desc("total_defects")).show()
#4
df.groupBy("shift").agg(F.round(F.avg("downtime_minutes"), 2).alias("avg_downtime")).orderBy(F.desc("avg_downtime")).show()
#5
df.groupBy("product_family").agg(F.round(F.avg("energy_kwh"), 2).alias("avg_energy_kwh")).orderBy(F.desc("avg_energy_kwh")).show()
#6
df.groupBy("quality_status").count().orderBy(F.desc("count")).show()
#7
df.groupBy("plant_id", "shift").agg(F.sum("units_produced").alias("total_production")).orderBy("plant_id", F.desc("total_production")).show(20)
#8
df.groupBy("plant_id", "product_family").agg(F.sum("defective_units").alias("total_defects")).orderBy("plant_id", F.desc("total_defects")).show(30)
#9
df.groupBy("machine_id").agg(F.round(F.avg("temperature_c"), 2).alias("avg_temperature")).orderBy(F.desc("avg_temperature")).show(30)
#10
df.groupBy("supervisor").agg(F.sum("units_produced").alias("total_output")).orderBy(F.desc("total_output")).show()

+--------+----------------+
|plant_id|total_production|
+--------+----------------+
|    PL03|        13547769|
|    PL02|        13546452|
|    PL04|        13529935|
|    PL06|        13492337|
|    PL01|        13488232|
|    PL05|        13323163|
+--------+----------------+

+--------------+----------------+
|product_family|total_production|
+--------------+----------------+
|        Valves|        20433082|
|         Pumps|        20208739|
|   Controllers|        20168572|
|        Motors|        20117495|
+--------------+----------------+

+--------+-------------+
|plant_id|total_defects|
+--------+-------------+
|    PL01|       339720|
|    PL02|       339647|
|    PL03|       339264|
|    PL06|       338993|
|    PL04|       338118|
|    PL05|       334502|
+--------+-------------+

+-------+------------+
|  shift|avg_downtime|
+-------+------------+
|Evening|       34.22|
|  Night|       33.95|
|Morning|       33.62|
+-------+------------+

+--------------+--------------+
|

## 10. Joining DataFrames

In [13]:
plant_df = spark.read.csv("plant_master.csv", header=True, inferSchema=True)
machine_df = spark.read.csv("machine_master.csv", header=True, inferSchema=True)

#1 inner join production + plant master
prod_plant_df = df.join(plant_df, on="plant_id", how="inner")
prod_plant_df.show()
print(prod_plant_df.count())
#2
prod_plant_df.select("production_id", "plant_name", "machine_id", "units_produced").show()
#3 (drop machine_master's plant_id to avoid a duplicate column)
machine_slim_df = machine_df.drop("plant_id")
prod_machine_df = df.join(machine_slim_df, on="machine_id", how="inner")
prod_machine_df.select("production_id", "machine_id", "machine_type", "capacity_per_hour", "units_produced").show()
#4 production + plant + machine
full_df = df.join(plant_df, on="plant_id", how="inner").join(machine_slim_df, on="machine_id", how="inner")
full_df.printSchema()
print(full_df.count())
#5
full_df.groupBy("city").agg(F.sum("units_produced").alias("total_production")).orderBy(F.desc("total_production")).show()
#6
full_df.groupBy("machine_type").agg(F.round(F.avg("downtime_minutes"), 2).alias("avg_downtime")).orderBy(F.desc("avg_downtime")).show()
#7
full_df.groupBy("region").agg(F.sum("defective_units").alias("total_defects")).orderBy(F.desc("total_defects")).show()
#8 production records with NO matching machine master record
no_machine_df = df.join(machine_df, on="machine_id", how="left_anti")
no_machine_df.show()
print(no_machine_df.count())
#9 left join and inspect unmatched rows
left_df = df.join(machine_slim_df, on="machine_id", how="left")
unmatched_df = left_df.filter(F.col("machine_type").isNull())
unmatched_df.show()
print(unmatched_df.count())
#10
full_df.groupBy("plant_name").agg(F.round(F.sum("energy_kwh"), 2).alias("total_energy_kwh")).orderBy(F.desc("total_energy_kwh")).show()

+--------+-------------+---------------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------+---------+------+-----------+-------------+
|plant_id|production_id|production_date|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|     plant_name|     city|region|year_opened|plant_manager|
+--------+-------------+---------------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+---------------+---------+------+-----------+-------------+
|    PL03|    PR0000001|     2026-08-07|     MC011|     OP0275|Evening|     MTR-100|    

## 11. Updating Data
Spark DataFrames are immutable — every "update" returns a new DataFrame.

In [14]:
df_upd = df

#1
df_upd = df_upd.withColumn("quality_status", F.when(F.col("quality_status") == "Reject", "Failed").otherwise(F.col("quality_status")))
df_upd.groupBy("quality_status").count().show()
#2
df_upd = df_upd.withColumn("quality_status", F.when(F.col("quality_status") == "Rework", "Requires Review").otherwise(F.col("quality_status")))
df_upd.groupBy("quality_status").count().show()
#3
df_upd = df_upd.withColumn(
    "units_planned",
    F.when(F.col("product_family") == "Motors", F.round(F.col("units_planned") * 1.05).cast("int")).otherwise(F.col("units_planned"))
)
df_upd.groupBy("product_family").agg(F.sum("units_planned").alias("total_planned")).show()
#4
df_upd = df_upd.withColumn(
    "energy_kwh",
    F.when(F.col("shift") == "Night", F.round(F.col("energy_kwh") * 1.03, 2)).otherwise(F.col("energy_kwh"))
)
df_upd.groupBy("shift").agg(F.round(F.sum("energy_kwh"), 2).alias("total_energy")).show()
#5
df_upd = df_upd.fillna({"operator_id": "UNASSIGNED"})
print(df_upd.filter(F.col("operator_id") == "UNASSIGNED").count())
#6
df_upd = df_upd.withColumn("supervisor", F.when(F.col("supervisor") == "Anita", "Anita Rao").otherwise(F.col("supervisor")))
df_upd.groupBy("supervisor").count().show()
#7 corrected temperature = own value, else that machine's average
machine_window = Window.partitionBy("machine_id")
df_upd = df_upd.withColumn(
    "temperature_corrected",
    F.coalesce(F.col("temperature_c"), F.round(F.avg("temperature_c").over(machine_window), 2))
)
df_upd.select("production_id", "machine_id", "temperature_c", "temperature_corrected") \
    .filter(F.col("temperature_c").isNull()).show()
print(df_upd.filter(F.col("temperature_corrected").isNull()).count())

+--------------+-----+
|quality_status|count|
+--------------+-----+
|        Rework|22928|
|        Failed|22744|
|          Pass|74328|
+--------------+-----+

+---------------+-----+
| quality_status|count|
+---------------+-----+
|         Failed|22744|
|Requires Review|22928|
|           Pass|74328|
+---------------+-----+

+--------------+-------------+
|product_family|total_planned|
+--------------+-------------+
|         Pumps|     21744889|
|        Valves|     21986559|
|   Controllers|     21707690|
|        Motors|     22739080|
+--------------+-------------+

+-------+------------+
|  shift|total_energy|
+-------+------------+
|Evening|  9410984.47|
|Morning|  9405926.53|
|  Night|   9837637.4|
+-------+------------+

1524
+----------+-----+
|supervisor|count|
+----------+-----+
|     Farah|15238|
|    Harini|15080|
|     Irfan|14849|
|    Bharat|15112|
|    Charan|14930|
|     Gopal|14875|
|     Deepa|15027|
| Anita Rao|14889|
+----------+-----+

+-------------+---------

## 12. Null Values and na Functions

In [15]:
#1
print(df.filter(F.col("operator_id").isNull()).count())
#2
print(df.filter(F.col("temperature_c").isNull()).count())
#3
print(df.filter(F.col("vibration_mm_s").isNull()).count())
#4
print(df.filter(F.col("downtime_minutes").isNull()).count())
# missing values in every column at once
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()
#5
df_dropped_all = df.na.drop()
print(df_dropped_all.count())
#6
df_temp_dropped = df.na.drop(subset=["temperature_c"])
print(df_temp_dropped.count())
#7
df_downtime_filled = df.na.fill({"downtime_minutes": 0})
print(df_downtime_filled.filter(F.col("downtime_minutes").isNull()).count())
#8
df_operator_filled = df.fillna({"operator_id": "UNKNOWN"})
print(df_operator_filled.filter(F.col("operator_id") == "UNKNOWN").count())
#9
avg_temp = df.select(F.round(F.avg("temperature_c"), 2)).first()[0]
print("Average temperature:", avg_temp)
df_temp_filled = df.na.fill({"temperature_c": avg_temp})
print(df_temp_filled.filter(F.col("temperature_c").isNull()).count())
#10
df_clean = df.na.fill({"operator_id": "UNKNOWN", "downtime_minutes": 0, "temperature_c": avg_temp})
print("Original rows:            ", df.count())
print("After na.drop():          ", df_dropped_all.count())
print("After dropping temp only: ", df_temp_dropped.count())
print("After filling (cleansed): ", df_clean.count())

# also practice: isNotNull() and coalesce()
print(df.filter(F.col("temperature_c").isNotNull()).count())
df.select("production_id", "downtime_minutes", F.coalesce(F.col("downtime_minutes"), F.lit(0)).alias("downtime_filled")) \
    .filter(F.col("downtime_minutes").isNull()).show()
df.select("production_id", F.coalesce(F.col("temperature_c"), F.lit(avg_temp)).alias("temperature_final")).show()

1524
2132
1788
1177
+-------------+---------------+--------+----------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-----+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|            0|              0|       0|         0|       1524|    0|           0|             0|            0|             0|              0|            1177|         0|         2132|          1788|             0|               0|         0|
+-------

## 13. Built-in Functions

In [16]:
df_fn = df

#1
df_fn = df_fn.withColumn("product_family_upper", F.upper("product_family"))
df_fn.select("product_family", "product_family_upper").show()
#2
df_fn = df_fn.withColumn("supervisor_lower", F.lower("supervisor"))
df_fn.select("supervisor", "supervisor_lower").show()
#3
df_fn = df_fn.withColumn("operator_id_trimmed", F.trim("operator_id"))
df_fn.select("operator_id", "operator_id_trimmed").show()
#4
df_fn = df_fn.withColumn("product_code_length", F.length("product_code"))
df_fn.select("product_code", "product_code_length").show()
#5 concat_ws (with separator) and concat (without)
df_fn = df_fn.withColumn("plant_machine", F.concat_ws("-", "plant_id", "machine_id")) \
             .withColumn("plant_machine_concat", F.concat("plant_id", F.lit("_"), "machine_id"))
df_fn.select("plant_id", "machine_id", "plant_machine", "plant_machine_concat").show()
#6
df_fn = df_fn.withColumn("temperature_1dp", F.round("temperature_c", 1))
df_fn.select("temperature_c", "temperature_1dp").show()
#7
df_fn = df_fn.withColumn("unit_gap", F.abs(F.col("units_planned") - F.col("units_produced")))
df_fn.select("units_planned", "units_produced", "unit_gap").show()
#8
df_fn = df_fn.withColumn("production_id_new", F.regexp_replace("production_id", "PR", "PROD-"))
df_fn.select("production_id", "production_id_new").show()
#9
df_fn = df_fn.withColumn("code_prefix", F.substring("product_code", 1, 3)) \
             .withColumn("code_number", F.substring("product_code", 5, 3))
df_fn.select("product_code", "code_prefix", "code_number").show()
#10
df_fn = df_fn.withColumn("temperature_final", F.coalesce(F.col("temperature_c"), F.lit(avg_temp)))
df_fn.select("temperature_c", "temperature_final").filter(F.col("temperature_c").isNull()).show()

+--------------+--------------------+
|product_family|product_family_upper|
+--------------+--------------------+
|        Motors|              MOTORS|
|   Controllers|         CONTROLLERS|
|        Motors|              MOTORS|
|         Pumps|               PUMPS|
|         Pumps|               PUMPS|
|        Motors|              MOTORS|
|        Valves|              VALVES|
|        Valves|              VALVES|
|   Controllers|         CONTROLLERS|
|   Controllers|         CONTROLLERS|
|         Pumps|               PUMPS|
|   Controllers|         CONTROLLERS|
|   Controllers|         CONTROLLERS|
|        Valves|              VALVES|
|        Motors|              MOTORS|
|   Controllers|         CONTROLLERS|
|   Controllers|         CONTROLLERS|
|        Motors|              MOTORS|
|        Valves|              VALVES|
|         Pumps|               PUMPS|
+--------------+--------------------+
only showing top 20 rows
+----------+----------------+
|supervisor|supervisor_lower|
+--

## 14. when() / otherwise()

In [17]:
df_cat = df.withColumn("production_efficiency", F.col("units_produced") / F.col("units_planned") * 100)

# Efficiency Category
df_cat = df_cat.withColumn(
    "efficiency_category",
    F.when(F.col("production_efficiency") >= 100, "Above Target")
     .when(F.col("production_efficiency") >= 95, "On Target")
     .when(F.col("production_efficiency") >= 85, "Below Target")
     .otherwise("Critical")
)
df_cat.select("production_id", "production_efficiency", "efficiency_category").show()

# Defect Category
df_cat = df_cat.withColumn(
    "defect_category",
    F.when(F.col("defective_units") == 0, "Perfect")
     .when(F.col("defective_units").between(1, 10), "Normal")
     .when(F.col("defective_units").between(11, 25), "Attention")
     .otherwise("High Defect")
)
df_cat.select("production_id", "defective_units", "defect_category").show()

# Machine Condition (assumed thresholds: temperature > 75 or vibration > 6)
df_cat = df_cat.withColumn(
    "machine_condition",
    F.when((F.col("temperature_c") > 75) | (F.col("vibration_mm_s") > 6), "Inspect").otherwise("Normal")
)
df_cat.select("production_id", "temperature_c", "vibration_mm_s", "machine_condition").show()

# Count records in each category
df_cat.groupBy("efficiency_category").count().orderBy(F.desc("count")).show()
df_cat.groupBy("defect_category").count().orderBy(F.desc("count")).show()
df_cat.groupBy("machine_condition").count().orderBy(F.desc("count")).show()

+-------------+---------------------+-------------------+
|production_id|production_efficiency|efficiency_category|
+-------------+---------------------+-------------------+
|    PR0000001|    93.67924528301886|       Below Target|
|    PR0000002|   103.33745364647713|       Above Target|
|    PR0000003|    85.97285067873304|       Below Target|
|    PR0000004|     91.4866581956798|       Below Target|
|    PR0000005|    86.04651162790698|       Below Target|
|    PR0000006|    82.98538622129436|           Critical|
|    PR0000007|    81.97424892703863|           Critical|
|    PR0000008|   103.82830626450117|       Above Target|
|    PR0000009|    92.33226837060703|       Below Target|
|    PR0000010|    99.06015037593986|          On Target|
|    PR0000011|    88.73483535528595|       Below Target|
|    PR0000012|    98.77551020408163|          On Target|
|    PR0000013|    86.35458167330677|       Below Target|
|    PR0000014|   101.78571428571428|       Above Target|
|    PR0000015

## 15. Complex JSON

In [18]:
machine_json = spark.read.option("multiline", "true").json("machine_configurations.json")

#1
machine_json.printSchema()
#2
machine_json.select("machine_id", "machine_type").show()
#3
machine_json.select("machine_id", F.col("specifications.rated_power_kw").alias("rated_power_kw")).show()
#4
machine_json.select("machine_id", F.col("specifications.max_temperature_c").alias("max_temperature_c")).show()
#5
machine_json.select("machine_id", F.col("specifications.vibration_limit_mm_s").alias("vibration_limit_mm_s")).show()
#6
machine_json.select(
    "machine_id",
    F.col("maintenance.frequency_days").alias("maintenance_frequency_days"),
    F.col("maintenance.last_service").alias("last_service")
).show()
#7 working with the sensors array
machine_json.select(
    "machine_id",
    "sensors",
    F.size("sensors").alias("sensor_count"),
    F.col("sensors")[0]["sensor"].alias("first_sensor"),
    F.col("sensors.sensor").alias("sensor_names"),
    F.array_contains(F.col("sensors.sensor"), "temperature").alias("has_temperature_sensor")
).show()
#8
exploded_df = machine_json.select("machine_id", F.explode("sensors").alias("sensor_info"))
exploded_df.show()
#9 one row per machine / sensor
machine_sensor_df = machine_json.select(
    "machine_id", "machine_type", F.explode("sensors").alias("s")
).select(
    "machine_id", "machine_type",
    F.col("s.sensor").alias("sensor"),
    F.col("s.unit").alias("unit"),
    F.col("s.active").alias("active")
)
machine_sensor_df.show()
print(machine_sensor_df.count())
#10 machines with an active temperature sensor
machine_sensor_df.filter((F.col("sensor") == "temperature") & (F.col("active") == True)) \
    .select("machine_id", "machine_type").show()
#11 join machine configuration with production data
machine_cfg = machine_json.select(
    "machine_id", "machine_type", "capacity_per_hour", "installed_year",
    F.col("specifications.rated_power_kw").alias("rated_power_kw"),
    F.col("specifications.max_temperature_c").alias("max_temperature_c"),
    F.col("specifications.vibration_limit_mm_s").alias("vibration_limit_mm_s"),
    F.col("maintenance.frequency_days").alias("maintenance_frequency_days")
)
prod_cfg_df = df.join(machine_cfg, on="machine_id", how="left")
prod_cfg_df.select(
    "production_id", "machine_id", "machine_type", "temperature_c", "max_temperature_c",
    "vibration_mm_s", "vibration_limit_mm_s"
).show()
print(prod_cfg_df.count())
# bonus: runs where the machine exceeded its configured limits
prod_cfg_df.filter(
    (F.col("temperature_c") > F.col("max_temperature_c")) | (F.col("vibration_mm_s") > F.col("vibration_limit_mm_s"))
).count()

root
 |-- capacity_per_hour: long (nullable = true)
 |-- installed_year: long (nullable = true)
 |-- machine_id: string (nullable = true)
 |-- machine_type: string (nullable = true)
 |-- maintenance: struct (nullable = true)
 |    |-- frequency_days: long (nullable = true)
 |    |-- last_service: string (nullable = true)
 |-- plant_id: string (nullable = true)
 |-- sensors: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- active: boolean (nullable = true)
 |    |    |-- sensor: string (nullable = true)
 |    |    |-- unit: string (nullable = true)
 |-- specifications: struct (nullable = true)
 |    |-- max_temperature_c: long (nullable = true)
 |    |-- rated_power_kw: long (nullable = true)
 |    |-- vibration_limit_mm_s: double (nullable = true)

+----------+------------+
|machine_id|machine_type|
+----------+------------+
|     MC001|         CNC|
|     MC002|    Assembly|
|     MC003|     Casting|
|     MC004|     Welding|
|     MC005|     Test

12319

## 16. Union and Union ALL

In [19]:
extra_df = spark.read.csv("manufacturing_production_extra_12k.csv", header=True, inferSchema=True)

#1
main_count = df.count()
print(main_count)
#2
extra_count = extra_df.count()
print(extra_count)
#3 (union keeps duplicates in Spark — it behaves like SQL UNION ALL)
union_df = df.union(extra_df)
print(union_df.count())
#4
union_all_df = df.unionAll(extra_df)
print(union_all_df.count())
#5
print(main_count, extra_count, union_df.count(), union_all_df.count(), main_count + extra_count)
#6 duplicate production_id values
dup_ids = union_df.groupBy("production_id").count().filter(F.col("count") > 1)
dup_ids.show(10)
print(dup_ids.count())
#7
d_all_cols = union_df.dropDuplicates()
d_by_id = union_df.dropDuplicates(["production_id"])
#8
print("union:                    ", union_df.count())
print("dropDuplicates() all cols:", d_all_cols.count())
print("dropDuplicates(id):       ", d_by_id.count())
#9
by_name_df = df.unionByName(extra_df)
print(by_name_df.count())
#10 swap two columns in extra_df, then compare union vs unionByName
cols = extra_df.columns[:]
i, j = cols.index("plant_id"), cols.index("machine_id")
cols[i], cols[j] = cols[j], cols[i]
extra_swapped = extra_df.select(*cols)
print(extra_swapped.columns[:5])

wrong_df = df.union(extra_swapped)            # matches by POSITION -> mixes plant/machine values
print("distinct plant_id with union():       ", wrong_df.select("plant_id").distinct().count())
right_df = df.unionByName(extra_swapped)      # matches by NAME -> correct
print("distinct plant_id with unionByName(): ", right_df.select("plant_id").distinct().count())

120000
12500
132500
132500
120000 12500 132500 132500 132500
+-------------+-----+
|production_id|count|
+-------------+-----+
|    PR0000413|    2|
|    PR0000135|    2|
|    PR0000236|    2|
|    PR0000339|    2|
|    PR0000059|    2|
|    PR0000463|    2|
|    PR0000136|    2|
|    PR0000263|    2|
|    PR0000393|    2|
|    PR0000445|    2|
+-------------+-----+
only showing top 10 rows
500
union:                     132500
dropDuplicates() all cols: 132500
dropDuplicates(id):        132000
132500
['production_id', 'production_date', 'machine_id', 'plant_id', 'operator_id']
distinct plant_id with union():        36
distinct plant_id with unionByName():  6


## 17. Window Functions

In [20]:
w_plant_rank = Window.partitionBy("plant_id").orderBy(F.col("units_produced").desc())

#1, #2, #3, #4 rank, row_number, rank, dense_rank within each plant
ranked_df = df.withColumn("row_num", F.row_number().over(w_plant_rank)) \
              .withColumn("rank_no", F.rank().over(w_plant_rank)) \
              .withColumn("dense_rank_no", F.dense_rank().over(w_plant_rank))
ranked_df.select("plant_id", "production_id", "units_produced", "row_num", "rank_no", "dense_rank_no") \
    .orderBy("plant_id", "row_num").show(20)

#5 top 5 production runs per plant
top5_df = ranked_df.filter(F.col("row_num") <= 5)
top5_df.select("plant_id", "production_id", "units_produced", "row_num").orderBy("plant_id", "row_num").show(30)

#6 top 3 machines by output within each plant
machine_totals = df.groupBy("plant_id", "machine_id").agg(F.sum("units_produced").alias("total_output"))
w_machine_rank = Window.partitionBy("plant_id").orderBy(F.col("total_output").desc())
machine_totals.withColumn("machine_rank", F.row_number().over(w_machine_rank)) \
    .filter(F.col("machine_rank") <= 3).orderBy("plant_id", "machine_rank").show(18)

#7 total plant production without collapsing rows
w_plant = Window.partitionBy("plant_id")
df.withColumn("plant_total_output", F.sum("units_produced").over(w_plant)) \
    .select("production_id", "plant_id", "units_produced", "plant_total_output").show()

#8 average machine output using a window
w_machine = Window.partitionBy("machine_id")
df.withColumn("machine_avg_output", F.round(F.avg("units_produced").over(w_machine), 2)) \
    .select("production_id", "machine_id", "units_produced", "machine_avg_output").show()

#9 cumulative production by machine ordered by date
w_cumulative = Window.partitionBy("machine_id") \
    .orderBy("production_date", "production_id") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
cumulative_df = df.withColumn("cumulative_output", F.sum("units_produced").over(w_cumulative))
cumulative_df.filter(F.col("machine_id") == "MC001") \
    .select("machine_id", "production_date", "production_id", "units_produced", "cumulative_output") \
    .orderBy("production_date", "production_id").show()

#10 lag, #11 lead, #12 change from previous run
w_seq = Window.partitionBy("machine_id").orderBy("production_date", "production_id")
seq_df = df.withColumn("previous_output", F.lag("units_produced", 1).over(w_seq))
seq_df = seq_df.withColumn("next_output", F.lead("units_produced", 1).over(w_seq))
seq_df = seq_df.withColumn("change_from_previous", F.col("units_produced") - F.col("previous_output"))
seq_df.filter(F.col("machine_id") == "MC001") \
    .select("production_date", "production_id", "units_produced", "previous_output", "next_output", "change_from_previous") \
    .orderBy("production_date", "production_id").show()

#13 rank product families by total output
family_totals = df.groupBy("product_family").agg(F.sum("units_produced").alias("total_output"))
w_family = Window.orderBy(F.col("total_output").desc())
family_totals.withColumn("family_rank", F.rank().over(w_family)).show()

+--------+-------------+--------------+-------+-------+-------------+
|plant_id|production_id|units_produced|row_num|rank_no|dense_rank_no|
+--------+-------------+--------------+-------+-------+-------------+
|    PL01|    PR0073059|          1239|      1|      1|            1|
|    PL01|    PR0002678|          1238|      2|      2|            2|
|    PL01|    PR0005192|          1236|      3|      3|            3|
|    PL01|    PR0044516|          1236|      4|      3|            3|
|    PL01|    PR0056319|          1236|      5|      3|            3|
|    PL01|    PR0006350|          1234|      6|      6|            4|
|    PL01|    PR0046906|          1233|      7|      7|            5|
|    PL01|    PR0046582|          1230|      8|      8|            6|
|    PL01|    PR0024941|          1229|      9|      9|            7|
|    PL01|    PR0088933|          1229|     10|      9|            7|
|    PL01|    PR0097172|          1229|     11|      9|            7|
|    PL01|    PR0048

## 18. Date and Time Functions

In [21]:
#1
df_dt = df.withColumn("production_date", F.to_date("production_date", "yyyy-MM-dd"))
df_dt.select("production_id", "production_date").printSchema()
df_dt.select("production_id", "production_date").show()
#2, #3, #4
df_dt = df_dt.withColumn("year", F.year("production_date")) \
             .withColumn("month", F.month("production_date")) \
             .withColumn("day", F.dayofmonth("production_date"))
df_dt.select("production_date", "year", "month", "day").show()
#5
df_dt = df_dt.withColumn("month_name", F.date_format("production_date", "MMMM"))
df_dt.select("production_date", "month_name").show()
#6
df_dt = df_dt.withColumn("day_name", F.date_format("production_date", "EEEE"))
df_dt.select("production_date", "day_name").show()
#7
df_dt = df_dt.withColumn("year_month", F.date_format("production_date", "yyyy-MM"))
df_dt.select("production_date", "year_month").show()
#8
df_dt.groupBy("year_month").agg(F.sum("units_produced").alias("total_production")).orderBy("year_month").show()
#9
df_dt.groupBy("year_month").agg(F.sum("defective_units").alias("total_defects")).orderBy("year_month").show()
#10
df_dt.groupBy("year_month", "plant_id").agg(F.sum("units_produced").alias("total_production")).orderBy("year_month", "plant_id").show()
#11 production between two dates
march_df = df_dt.filter(F.col("production_date").between(F.to_date(F.lit("2026-03-01")), F.to_date(F.lit("2026-03-31"))))
print(march_df.count())
march_df.agg(F.sum("units_produced").alias("march_production")).show()
#12
df_dt = df_dt.withColumn("date_plus_7", F.date_add("production_date", 7))
df_dt.select("production_date", "date_plus_7").show()
#13
df_dt = df_dt.withColumn("date_minus_30", F.date_sub("production_date", 30))
df_dt.select("production_date", "date_minus_30").show()
#14
df_dt = df_dt.withColumn("days_since_production", F.datediff(F.current_date(), "production_date"))
df_dt.select("production_date", "days_since_production").show()
#15
df_dt.groupBy("year_month").agg(F.sum("units_produced").alias("total_production")) \
    .orderBy(F.desc("total_production")).limit(1).show()

root
 |-- production_id: string (nullable = true)
 |-- production_date: date (nullable = true)

+-------------+---------------+
|production_id|production_date|
+-------------+---------------+
|    PR0000001|     2026-08-07|
|    PR0000002|     2026-08-03|
|    PR0000003|     2026-05-20|
|    PR0000004|     2026-01-15|
|    PR0000005|     2026-09-21|
|    PR0000006|     2026-09-03|
|    PR0000007|     2026-05-07|
|    PR0000008|     2026-08-17|
|    PR0000009|     2026-03-10|
|    PR0000010|     2026-02-13|
|    PR0000011|     2026-09-29|
|    PR0000012|     2026-07-10|
|    PR0000013|     2026-07-19|
|    PR0000014|     2026-05-05|
|    PR0000015|     2026-03-15|
|    PR0000016|     2026-03-03|
|    PR0000017|     2026-08-04|
|    PR0000018|     2026-01-30|
|    PR0000019|     2026-01-11|
|    PR0000020|     2026-02-18|
+-------------+---------------+
only showing top 20 rows
+---------------+----+-----+---+
|production_date|year|month|day|
+---------------+----+-----+---+
|     2026-0